# Assignment 1: Gaussians, Categories, and Clusters — SOLUTION KEY (GenJAX)

**Instructor answer key — do not distribute to students.**

This is the worked solution for the GenJAX (canonical) stencil `clusters.ipynb`.
Every `# fill me` cell is replaced with a `# SOLUTION` implementation, and the
derivation / discussion cells contain model answers.

**Corresponding textbook chapters:** Tutorial 1 Ch 5 (Bayesian inference) and
[Tutorial 3 Ch 5 — Mixture Models](https://josephausterweil.github.io/probintro/intro2/05_mixture_models/).

---

## Setup

If running in Google Colab, run the cell below to install GenJAX. (On the first run of the session only.)

In [ ]:
!pip install genjax

In [ ]:
# Import packages
import jax
import jax.numpy as jnp
import jax.random as random
import jax.lax as lax
from genjax import gen, flip, normal   # NOTE: `flip(p)` takes a probability; `bernoulli(p)` takes a logit.
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

np.random.seed(42)
key = random.PRNGKey(42)

# NOTE on dtypes: GenJAX distributions produce float32. When you pass scalars
# into a GenJAX model (Part 2(e)), cast them with jnp.float32(...) so they do
# not mix with numpy's float64 — a dtype mismatch raises a TypeError inside the
# sampler. The Part 2(e) solution below does this.

---

# Problem 1: Gaussian-Gaussian Conjugate Model

We start with the **Gaussian-Gaussian** conjugate model: a Gaussian likelihood with unknown mean $\mu$ and *known* variance $\sigma_x^2$, with a Gaussian prior on $\mu$:

$$\mu \sim \mathcal{N}(\mu_0, \sigma_0^2) \qquad \qquad x_1, \dotsc, x_N \mid \mu, \sigma_x^2 \overset{iid}{\sim} \mathcal{N}(\mu, \sigma_x^2)$$

Because the prior is conjugate to the likelihood, the posterior and posterior-predictive distributions have closed forms:

$$\mu \mid x_1, \dotsc, x_N \sim \mathcal{N}\!\left(\frac{\mu_0 \sigma_0^{-2} + \sigma_x^{-2} \sum_n x_n}{\sigma_0^{-2} + N \sigma_x^{-2}},\; \left[\sigma_0^{-2} + N \sigma_x^{-2}\right]^{-1}\right)$$

$$x_{N+1} \mid x_1, \dotsc, x_N \sim \mathcal{N}\!\left(\text{same mean as posterior},\; \left[\sigma_0^{-2} + N \sigma_x^{-2}\right]^{-1} + \sigma_x^2\right)$$

The predictive has the same mean as the posterior but its variance is inflated by $\sigma_x^2$ — the irreducible noise of the likelihood.

**For Problem 1, use $\mu_0 = 0$ and $\sigma_0^2 = 1$.**

**Note on GenJAX.** Problem 1 is closed-form, so we will use `numpy` + `scipy.stats` here — there is nothing for GenJAX to add. GenJAX returns in Problem 2 Part (e), where we sample from a non-conjugate mixture model.


## Part 1(a): Prior plot

To provide a baseline, plot the prior distribution $p(\mu) = \mathcal{N}(\mu; \mu_0, \sigma_0^2)$ over a range that captures both tails and the peak.


In [ ]:
# SOLUTION
from scipy.stats import norm

mu_0 = 0.0
sigma_0_squared = 1.0
mu_range = np.linspace(-4, 4, 1000)

prior_density = norm.pdf(mu_range, mu_0, np.sqrt(sigma_0_squared))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(mu_range, prior_density, color='C0', lw=2, label=r'$p(\mu)$')
ax.fill_between(mu_range, prior_density, alpha=0.15, color='C0')

ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$p(\mu)$')
ax.set_title(r'Prior: $\mathcal{N}(\mu_0, \sigma_0^2)$')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## Part 1(b): One-datum update

Calculate and plot the **posterior** $p(\mu \mid x_1)$ **and** the **posterior-predictive** $p(x_2 \mid x_1)$ after observing $x_1 = 2$, for $\sigma_x^2 = 0.25$ and $\sigma_x^2 = 4$. That is four distributions in total (posterior + predictive for each $\sigma_x^2$).

**Question.** How does changing the likelihood variance $\sigma_x^2$ affect the posterior and the predictive? Where are the two distributions similar? Where do they differ, and why?

**Hint.** A pre-written helper `conjugate_update` is provided below. It returns the posterior mean and variance, and the posterior-predictive variance. Fill in its body using the closed-form expressions above.


In [ ]:
# SOLUTION
def conjugate_update(mu_0, sigma_0_squared, sigma_x_squared, data):
    """
    Conjugate Gaussian-Gaussian update.

    Returns (post_mean, post_var, pred_var).
    """
    data = np.atleast_1d(np.asarray(data, dtype=float))
    N = data.size
    sum_x = data.sum()

    # Posterior precision = prior precision + N * likelihood precision.
    posterior_precision = 1.0 / sigma_0_squared + N / sigma_x_squared

    # Precision-weighted average of prior mean and data.
    post_mean = (mu_0 / sigma_0_squared + sum_x / sigma_x_squared) / posterior_precision
    post_var = 1.0 / posterior_precision

    # Predictive inflates the posterior variance by the irreducible noise sigma_x^2.
    pred_var = post_var + sigma_x_squared

    return post_mean, post_var, pred_var


In [ ]:
# SOLUTION
mu_0 = 0.0
sigma_0_squared = 1.0
x_1 = np.array([2.0])

sigma_x_squared_values = [0.25, 4.0]
plot_range = np.linspace(-4, 6, 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, sigma_x_squared in zip(axes, sigma_x_squared_values):
    post_mean, post_var, pred_var = conjugate_update(
        mu_0, sigma_0_squared, sigma_x_squared, x_1)

    posterior = norm.pdf(plot_range, post_mean, np.sqrt(post_var))
    predictive = norm.pdf(plot_range, post_mean, np.sqrt(pred_var))

    ax.plot(plot_range, posterior, color='C0', lw=2,
            label=f'posterior  (var={post_var:.3f})')
    ax.plot(plot_range, predictive, color='C1', lw=2,
            label=f'predictive (var={pred_var:.3f})')
    ax.axvline(x_1[0], color='k', ls='--', alpha=0.5, label=r'$x_1 = 2$')

    ax.set_xlabel('value')
    ax.set_ylabel('density')
    ax.set_title(rf'$\sigma_x^2 = {sigma_x_squared}$')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


**Your answer.**

The posterior and the predictive are **always centered at the same posterior
mean**, but the predictive is wider — its variance is exactly $\sigma_x^2$
larger, because a future observation carries the irreducible likelihood noise
on top of our remaining uncertainty about $\mu$.

Increasing $\sigma_x^2$ from 0.25 to 4 has two effects:

- The **posterior moves less** toward $x_1 = 2$. With $\sigma_x^2 = 0.25$ the
  likelihood is sharp, so the data dominates and the posterior mean shifts to
  $\approx 1.6$. With $\sigma_x^2 = 4$ the likelihood is diffuse, the prior
  wins, and the posterior mean stays near $\approx 0.4$.
- The **predictive becomes much wider**, because $\sigma_x^2$ is added directly
  to its variance ($0.45$ vs. $\approx 4.8$).

So the likelihood variance moves the posterior *less* but the predictive *more*.


## Part 1(c): Multiple-datum update

Now observe five data points: $(x_1, \dotsc, x_5) = (2.1, 2.5, 1.4, 2.2, 1.8)$. Plot the posterior and predictive for $\sigma_x^2 = 0.25$ and $\sigma_x^2 = 4$.

The average of these five points is exactly $2.0$, the same as the single datum in Part 1(b). **Compare** the resulting posteriors and predictives to Part 1(b).

**Question.** Where do the Part 1(b) and Part 1(c) results agree, and where do they differ? For the cases that differ, why? For the cases that don't, why not? (Hint: think about the role of $N$ in the posterior precision.)


In [ ]:
# SOLUTION
mu_0 = 0.0
sigma_0_squared = 1.0
data = np.array([2.1, 2.5, 1.4, 2.2, 1.8])

sigma_x_squared_values = [0.25, 4.0]
plot_range = np.linspace(-4, 6, 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, sigma_x_squared in zip(axes, sigma_x_squared_values):
    post_mean, post_var, pred_var = conjugate_update(
        mu_0, sigma_0_squared, sigma_x_squared, data)

    posterior = norm.pdf(plot_range, post_mean, np.sqrt(post_var))
    predictive = norm.pdf(plot_range, post_mean, np.sqrt(pred_var))

    ax.plot(plot_range, posterior, color='C0', lw=2,
            label=f'posterior  (var={post_var:.3f})')
    ax.plot(plot_range, predictive, color='C1', lw=2,
            label=f'predictive (var={pred_var:.3f})')
    ax.axvline(data.mean(), color='k', ls='--', alpha=0.5,
               label=f'data mean = {data.mean():.1f}')

    ax.set_xlabel('value')
    ax.set_ylabel('density')
    ax.set_title(rf'$\sigma_x^2 = {sigma_x_squared},\ N = 5$')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


**Your answer.**

The data mean is $2.0$ in both Part 1(b) and Part 1(c), so the posterior
**means** are similar in the two parts — what changes is the **variance**.

With $N = 5$ the posterior precision is $1/\sigma_0^2 + 5/\sigma_x^2$, so the
data term is five times larger than in the one-datum case. The posterior is
therefore **noticeably tighter**:

- For $\sigma_x^2 = 0.25$: data precision $= 5/0.25 = 20$ swamps the prior
  precision of $1$. Posterior variance $\approx 1/21 \approx 0.048$ — much
  sharper than Part 1(b).
- For $\sigma_x^2 = 4$: data precision $= 5/4 = 1.25$, now *comparable* to the
  prior. The posterior mean shifts further toward $2.0$ than in Part 1(b),
  and the variance shrinks to $\approx 1/2.25 \approx 0.44$.

The **predictive variance** behaves differently: it is dominated by
$\sigma_x^2$, not by the posterior variance. No matter how much data we
collect, the predictive variance asymptotes to $\sigma_x^2$ — collecting data
sharpens our belief about $\mu$ but never removes the per-observation noise.


---

# Problem 2: Gaussian Mixture / Categorization

In this problem, we make **categorization decisions** for two categories, each defined as a Gaussian distribution. You will derive the probability of an item being in one category vs. the other, then explore how the variances and prior probability of each category affect the posterior and the predictive distribution.

Data are generated by first picking which of two categories $c = 1, 2$ a datum belongs to (according to their prior probability) and then generating the datum from the corresponding category's likelihood:

$$c_n | \theta \sim \text{Bernoulli}(\theta) \qquad \qquad x_n | \mu_{c(n)}, \sigma_{c(n)}^2 \overset{iid}{\sim} \mathcal{N}(\mu_{c(n)}, \sigma_{c(n)}^2)$$

Note that $c(n)$ is the same as $c_n$; the parentheses are used to avoid double-subscripts. $c_n = 1$ with probability $\theta$ (and $c_n = 2$ with probability $1 - \theta$), so the prior probability of category 1 is $\theta$: $P(c_n = 1) = \theta$.

**For all of Problem 2, assume $\mu_1 = -1$ and $\mu_2 = 1$.**

---

## Part 2(a): Derivation — Categorization

Using **Bayes' rule**, derive the probability of a single datum being in category 1: $P(c_1 = 1 | x_1)$. You can assume that the values of $\mu_1, \mu_2, \sigma_1^2,$ and $\sigma_2^2$ are given parameters. Show your work (if you do not know how to create equations on a computer, you can scan your handwritten derivation and include it as an image).

As the next problem depends on this answer, the derivation should end up with:

$$P(c_1 = 1 | x_1) = \frac{\theta \, \mathcal{N}(x_1; \mu_1, \sigma_1^2)}{\theta \, \mathcal{N}(x_1; \mu_1, \sigma_1^2) + (1 - \theta) \, \mathcal{N}(x_1; \mu_2, \sigma_2^2)}$$

where $\mathcal{N}(x; \mu, \sigma^2)$ is the probability density of $x$ from a Normal distribution with mean $\mu$ and variance $\sigma^2$.

**Grading note: no credit for transcribing the given answer. Show every step (Bayes' rule, expand the denominator with the Law of Total Probability).**

#### **Derivation**

By **Bayes' rule**, with the category prior $P(c_1 = 1) = \theta$ and the
likelihood $p(x_1 \mid c_1 = 1) = \mathcal{N}(x_1; \mu_1, \sigma_1^2)$:

$$P(c_1 = 1 \mid x_1) = \frac{p(x_1 \mid c_1 = 1)\, P(c_1 = 1)}{p(x_1)}
= \frac{\theta\, \mathcal{N}(x_1; \mu_1, \sigma_1^2)}{p(x_1)}.$$

The denominator $p(x_1)$ is found by the **Law of Total Probability**, summing
over the two possible categories:

$$p(x_1) = \sum_{c} p(x_1 \mid c)\, P(c)
= \theta\, \mathcal{N}(x_1; \mu_1, \sigma_1^2)
+ (1-\theta)\, \mathcal{N}(x_1; \mu_2, \sigma_2^2).$$

Substituting the denominator back in:

$$P(c_1 = 1 \mid x_1) = \frac{\theta\, \mathcal{N}(x_1; \mu_1, \sigma_1^2)}
{\theta\, \mathcal{N}(x_1; \mu_1, \sigma_1^2)
+ (1-\theta)\, \mathcal{N}(x_1; \mu_2, \sigma_2^2)}.$$


---

## Part 2(b): Categorization

Calculate and plot the probability of being in category 1 (x-axis is the $x_1$ value; y-axis is $P(c_1 = 1 | x_1)$) for:

1. $\theta = 0.5$ and $\theta = 0.75$, with $\sigma_1^2 = \sigma_2^2 = 1$.
2. $\theta = 0.5$ when $\sigma_1^2 = 0.5$ and $\sigma_2^2 = 2$.
3. $\theta = 0.75$ when $\sigma_1^2 = 0.5$ and $\sigma_2^2 = 2$.

Make sure your plots capture the interesting behavior (appropriate x- and y-axis ranges).

**Hint.** A pre-written helper function `update_datum_c1` is provided below. It returns the posterior $P(c_1 = 1 | x)$ and the marginal $p(x)$ over an x-range. You should fill in its body using the result from part (a) — but use GenJAX primitives (see the suggestion comments inside). Alternatively, you may use `scipy.stats.norm.pdf` if you prefer to compute analytically; the tutorial chapters demonstrate both.

**Question.** Describe the effect of changing the prior and the variance on categorization decisions. Do they have the same effect? Why or why not?

In [ ]:
# SOLUTION
def update_datum_c1(mu_1, mu_2, sigma_1_squared, sigma_2_squared, theta, x_min, x_max):
    """
    Compute posterior probability of category 1 and the marginal distribution.

    Returns (x_range, posterior_c1, marginal).
    """
    x_range = np.linspace(x_min, x_max, 1000)

    # Component likelihoods.
    lik_1 = norm.pdf(x_range, mu_1, np.sqrt(sigma_1_squared))
    lik_2 = norm.pdf(x_range, mu_2, np.sqrt(sigma_2_squared))

    # Marginal p(x) via the Law of Total Probability.
    marginal = theta * lik_1 + (1.0 - theta) * lik_2

    # Posterior P(c=1|x) via Bayes' rule (Part 2(a)).
    posterior_c1 = (theta * lik_1) / marginal

    return x_range, posterior_c1, marginal


In [ ]:
# SOLUTION
mu_1 = -1.0
mu_2 = 1.0

theta_values = [0.5, 0.75]
configs = [
    (1, 1),      # sigma_1^2 = sigma_2^2 = 1
    (0.5, 2),    # sigma_1^2 = 0.5, sigma_2^2 = 2
]

colors = {0.5: "C0", 0.75: "C1"}
linestyles = {(1, 1): "-", (0.5, 2): "--"}

x_min, x_max = -6, 6

fig, ax = plt.subplots(figsize=(10, 6))

for theta in theta_values:
    for sigma_1_squared, sigma_2_squared in configs:
        x_range, posterior_c1, _ = update_datum_c1(
            mu_1, mu_2, sigma_1_squared, sigma_2_squared, theta, x_min, x_max)
        ax.plot(x_range, posterior_c1,
                color=colors[theta],
                linestyle=linestyles[(sigma_1_squared, sigma_2_squared)],
                lw=2,
                label=rf'$\theta={theta},\ \sigma_1^2={sigma_1_squared},\ \sigma_2^2={sigma_2_squared}$')

ax.axhline(0.5, color='k', alpha=0.3, lw=1)
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$P(c_1 = 1 | x)$')
ax.set_title('Posterior categorization probability')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


**Describe the effect of changing the prior and the variance on categorization decisions.**

Changing the prior $\theta$ and changing the variances do **not** have the same
effect.

- **Prior $\theta$**: increasing $\theta$ shifts the whole curve *upward* — it
  adds a constant offset to the log-odds $\log \frac{P(c=1|x)}{P(c=2|x)}$. With
  equal variances it simply moves the decision boundary (the $x$ where
  $P(c_1=1|x) = 0.5$) toward category 2, without changing the curve's shape.

- **Variances $\sigma_1^2, \sigma_2^2$**: these change the *shape* of the curve.
  The log-odds is a quadratic in $x$ when the variances differ, so the curve is
  no longer a simple monotone sigmoid — a smaller $\sigma_1^2$ makes category 1
  win sharply near $\mu_1$ but lose in the tails, because a tight Gaussian
  assigns very low density far from its mean.

So $\theta$ is an additive shift in log-odds; the variances reshape how fast
(and even in which direction) the log-odds changes with $x$.


---

## Part 2(c): Derivation — Prediction

Using Bayes' rule and the **Law of Total Probability**, derive the probability of a data point $p(x_1)$ according to this model (without any given data). As the next problem depends on this answer, the derivation should end up with:

$$p(x_1) = \theta \, \mathcal{N}(x_1; \mu_1, \sigma_1^2) + (1 - \theta) \, \mathcal{N}(x_1; \mu_2, \sigma_2^2)$$

**Grading note: no credit for transcribing the given answer. Show every step (Law of Total Probability over $c$, then expand $p(x|c)P(c)$ for each category).**

#### **Derivation**

The marginal $p(x_1)$ is obtained directly from the **Law of Total
Probability**, summing the joint $p(x_1, c)$ over the two categories:

$$p(x_1) = \sum_{c} p(x_1, c) = \sum_{c} p(x_1 \mid c)\, P(c).$$

Expanding the sum over $c \in \{1, 2\}$ with $P(c_1 = 1) = \theta$ and
$P(c_1 = 2) = 1 - \theta$:

$$p(x_1) = p(x_1 \mid c_1 = 1)\, P(c_1 = 1) + p(x_1 \mid c_1 = 2)\, P(c_1 = 2)$$

$$p(x_1) = \theta\, \mathcal{N}(x_1; \mu_1, \sigma_1^2)
+ (1-\theta)\, \mathcal{N}(x_1; \mu_2, \sigma_2^2).$$


---

## Part 2(d): Prediction

Plot $p(x_1)$ for:

1. $\theta = 0.5$ and $\theta = 0.75$, with $\sigma_1^2 = \sigma_2^2 = 1$.
2. $\theta = 0.5$ when $\sigma_1^2 = 0.5$ and $\sigma_2^2 = 2$.
3. $\theta = 0.75$ when $\sigma_1^2 = 0.5$ and $\sigma_2^2 = 2$.

Make sure your plots capture the interesting behavior.

Note that $p(x_1)$ is sometimes called the *marginal data distribution*; this type of model is called a **mixture model** because it composes a new distribution by "mixing" two (or more) distributions together.

**Question.** How does the prior and variance affect $p(x_1)$? Do they have the same effect? Why or why not?

In [ ]:
# SOLUTION
mu_1 = -1.0
mu_2 = 1.0

theta_values = [0.5, 0.75]
configs = [
    (1, 1),
    (0.5, 2),
]

colors = {0.5: "C0", 0.75: "C1"}
linestyles = {(1, 1): "-", (0.5, 2): "--"}

x_min, x_max = -6, 6

fig, ax = plt.subplots(figsize=(10, 6))

for theta in theta_values:
    for sigma_1_squared, sigma_2_squared in configs:
        x_range, _, marginal = update_datum_c1(
            mu_1, mu_2, sigma_1_squared, sigma_2_squared, theta, x_min, x_max)
        ax.plot(x_range, marginal,
                color=colors[theta],
                linestyle=linestyles[(sigma_1_squared, sigma_2_squared)],
                lw=2,
                label=rf'$\theta={theta},\ \sigma_1^2={sigma_1_squared},\ \sigma_2^2={sigma_2_squared}$')

ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$p(x)$')
ax.set_title('Marginal (predictive) distribution')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


**Describe the effect of changing the prior and the variance on $p(x_1)$.**

For the marginal $p(x_1)$ the prior and the variances again act differently:

- **Prior $\theta$** controls the relative *heights* of the two peaks. With
  $\theta = 0.5$ the two component peaks are equal; with $\theta = 0.75$ the
  category-1 peak (at $\mu_1 = -1$) is three times taller than the category-2
  peak.

- **Variances** control the *widths* of the two peaks. With equal
  $\sigma^2 = 1$ the mixture is cleanly bimodal. With $(\sigma_1^2, \sigma_2^2)
  = (0.5, 2)$ the category-1 component becomes a tall, narrow spike and
  category 2 a low, broad shoulder — and the visible bimodality can wash out
  into something that looks unimodal-with-skew.

They are not the same effect: $\theta$ redistributes probability *mass* between
the components, while the variances redistribute each component's mass *across
$x$*.


---

## Part 2(e): GenJAX Mixture Model

Write a GenJAX generative model `gaussian_mixture_model(theta, mu_1, mu_2, sigma_1, sigma_2)` that implements the generative process from the Problem Setup. Your model should:

1. Sample a category $c$ from a Bernoulli with parameter $\theta$, named `"category"`. **Use `flip(theta)`, not `bernoulli(theta)`** — see the note at the top of the notebook.
2. Conditional on $c$, sample $x$ from the corresponding Gaussian, named `"observation"`.
3. Return the observation $x$ and category $c$.

Then simulate $N = 2000$ samples with $\theta = 0.7$, $\mu_1 = -1, \mu_2 = 1$, $\sigma_1 = \sigma_2 = 1$. Plot a histogram of the observations overlaid on the theoretical marginal $p(x)$ from part (c). Verify that the empirical distribution matches.

**Hint.** Look at Tutorial 2, Chapter 2 for how to structure a `@gen` function. For the conditional Gaussian (sample from category 1's or category 2's likelihood based on $c$), `jnp.where(c, mu_1, mu_2)` and `jnp.where(c, sigma_1, sigma_2)` work cleanly because `flip` returns a Boolean that casts to 0/1.

In [ ]:
# SOLUTION

@gen
def gaussian_mixture_model(theta, mu_1, mu_2, sigma_1, sigma_2):
    # GenJAX generative model for a 2-component Gaussian mixture.
    # Sample the category with flip(theta): True = category 1, False = category 2.
    # flip() takes a probability directly (bernoulli() would take a logit).
    c = flip(theta) @ "category"

    # Pick the chosen component's parameters. jnp.where keeps the model traceable;
    # c casts to 1 (True) / 0 (False).
    mu_c = jnp.where(c, mu_1, mu_2)
    sigma_c = jnp.where(c, sigma_1, sigma_2)

    # Sample the observation from the chosen component.
    x = normal(mu_c, sigma_c) @ "observation"

    return x, c


# Simulate N = 2000 samples with theta = 0.7, mu_1 = -1, mu_2 = 1, sigma = 1.
# Cast model-argument scalars to float32 to match GenJAX's distribution dtype.
theta = jnp.float32(0.7)
mu_1, mu_2 = jnp.float32(-1.0), jnp.float32(1.0)
sigma_1, sigma_2 = jnp.float32(1.0), jnp.float32(1.0)
N = 2000

sim_keys = random.split(key, N)
traces = jax.vmap(
    lambda k: gaussian_mixture_model.simulate(k, (theta, mu_1, mu_2, sigma_1, sigma_2))
)(sim_keys)

x_samples, c_samples = traces.get_retval()

# Theoretical marginal p(x) from Part 2(c).
x_grid = np.linspace(-6, 6, 1000)
analytical_marginal = (
    theta * norm.pdf(x_grid, mu_1, sigma_1)
    + (1.0 - theta) * norm.pdf(x_grid, mu_2, sigma_2)
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(np.asarray(x_samples), bins=60, density=True, alpha=0.5,
        color='C0', label=f'simulated ({N} samples)')
ax.plot(x_grid, analytical_marginal, color='C3', lw=2,
        label='analytical p(x) from Part 2(c)')

ax.set_xlabel(r'$x$')
ax.set_ylabel('density')
ax.set_title('GenJAX mixture: empirical vs. analytical marginal')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

frac_cat1 = float(np.mean(np.asarray(c_samples)))
print(f"Fraction in category 1: {frac_cat1:.3f}  (expected {theta})")


---

## Submission

Submit this completed notebook (runs end-to-end with no errors) plus a short PDF with your derivations for parts (a) and (c) if you prefer to handwrite or typeset them separately.